In [ ]:
    ############    #############   Async Client Disconnection and Inference Cancellation   #############   ##############   

 =>  A user closes their browser tab mid-stream while your server is still generating
       tokens from an LLM -- if you don't detect that, you keep burning GPU time and tokens
       (money) on a response nobody will ever see.

 =>  FastAPI's 'await request.is_disconnected()' lets you detect this inside a streaming
       generator and cancel the upstream model call instead of letting it run to completion.

 =>  Pattern: run the token-generation loop, and on every iteration (or every N tokens)
       check for disconnection; if disconnected, break out and cancel the upstream task.


In [ ]:
import asyncio

async def generate_tokens(cancel_event: asyncio.Event):
    """Simulates an LLM streaming tokens one at a time."""
    tokens = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
    for token in tokens:
        if cancel_event.is_set():
            print("-- upstream generation cancelled, stopping early --")
            return
        await asyncio.sleep(0.1)
        yield token

async def simulate_client_disconnect_after(cancel_event: asyncio.Event, delay: float):
    await asyncio.sleep(delay)
    print("** client disconnected **")
    cancel_event.set()

async def main():
    cancel_event = asyncio.Event()
    disconnect_task = asyncio.create_task(
        simulate_client_disconnect_after(cancel_event, delay=0.35)
    )
    async for token in generate_tokens(cancel_event):
        print("streamed token:", token)
    await disconnect_task

await main()


In [ ]:
 =>  Streaming stops as soon as the (simulated) disconnect fires, instead of generating all
       9 tokens regardless of whether anyone is still listening.

 =>  In a real FastAPI StreamingResponse, replace 'cancel_event.is_set()' with
       'await request.is_disconnected()', and instead of just 'return', also cancel the real
       upstream task (e.g. an OpenAI/Anthropic streaming call) -- typically by catching
       asyncio.CancelledError around the upstream call -- so the provider stops billing
       you for tokens no client will receive.


In [ ]:
# Real FastAPI shape (reference only -- not executed here, no FastAPI app in this notebook):
#
# @app.get("/stream")
# async def stream(request: Request):
#     async def event_generator():
#         try:
#             async for token in call_llm_stream(prompt):
#                 if await request.is_disconnected():
#                     break
#                 yield f"data: {token}\n\n"
#         except asyncio.CancelledError:
#             pass  # upstream call cancelled cleanly
#     return StreamingResponse(event_generator(), media_type="text/event-stream")
print("see the commented reference pattern above")


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Build the real FastAPI SSE endpoint sketched above against a real (or mocked)
           streaming LLM call.

 =>  [ ] Manually disconnect mid-stream (close the browser tab / curl --max-time) and
           confirm, via logs, that the upstream call is actually cancelled, not left running.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Checking is_disconnected() only once at the start of the generator -- some ASGI
       servers only deliver the disconnect signal when you poll for it periodically inside
       the loop.

 =>  Cancelling the async generator but forgetting to also cancel/close the actual upstream
       HTTP call to the model provider -- the provider keeps generating (and billing) even
       though your server stopped reading the stream.
